# TensorRT Edge Inference Benchmark — Colab T4

Self-contained benchmark notebook for evaluating YOLOv8n under TensorRT FP32 / FP16 / INT8
precision constraints on a Colab T4 GPU. Produces `BenchmarkResult` JSON files in the same
schema as the Fedora CPU runs, enabling direct cross-runtime comparison.

**Before running — set runtime to T4 GPU:**
`Runtime → Change runtime type → T4 GPU`

**Files required on Google Drive** at `MyDrive/edge-inference-benchmark/`:
- `yolov8n.onnx` — exported locally via `python scripts/export_model.py`
- `calibration/` — 500-image INT8 calibration set (generated by `run_benchmark.py`)
- `val2017/` — COCO val2017 images (~1 GB)
- `annotations/` — COCO val2017 annotations

**Each cell is idempotent** — re-running any cell will not corrupt state.
Run cells top-to-bottom in order. Cell 7 (mAP evaluation) takes ~15 minutes total.

In [ ]:
# ── Cell 1 ── Environment Setup ───────────────────────────────────────────────
# Verify T4 GPU is available, install pinned packages, document runtime versions.
# Idempotent: pip skips already-installed versions.

import subprocess
import sys

# GPU check — must see Tesla T4
!nvidia-smi

# Install pinned versions
!pip install -q \
    onnx==1.16.0 \
    onnxruntime-gpu==1.18.0 \
    pycocotools==2.0.7 \
    numpy==1.26.4 \
    opencv-python-headless==4.10.0.84

import numpy as np
import torch
import onnxruntime as ort

# TensorRT and pycuda come pre-installed on Colab T4
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # creates CUDA context; must be imported before any cuda calls

# Hard assertion — this notebook cannot run without a GPU
assert torch.cuda.is_available(), (
    "GPU not available. Switch runtime: Runtime → Change runtime type → T4 GPU"
)

# TRT 8.6.x API is assumed throughout this notebook.
# Colab may ship a different version — if cells fail with AttributeError,
# check the version printed below and open an issue.
trt_major = int(trt.__version__.split(".")[0])
if trt_major != 8:
    print(f"WARNING: TensorRT {trt.__version__} detected; notebook targets 8.6.x. "
          "API differences may cause failures. See docs/benchmark-run-3-findings.md.")

COLAB_ENV = {
    "python":        sys.version.split()[0],
    "cuda":          torch.version.cuda,
    "torch":         torch.__version__,
    "tensorrt":      trt.__version__,
    "onnxruntime":   ort.__version__,
    "numpy":         np.__version__,
    "gpu":           torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2),
}

print("\n=== Colab Runtime Environment ===")
for k, v in COLAB_ENV.items():
    print(f"  {k:20s}: {v}")
print("\nEnvironment check passed.")

In [ ]:
# ── Cell 2 ── Mount Drive / Clone Repo / Verify ONNX Model ───────────────────
# Mount Drive, clone repo so src/ is importable, copy ONNX model to local disk.
# Idempotent: git pull is safe to re-run; model copy is guarded by existence check.

import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_URL  = "https://github.com/KNakul242/edge-inference-benchmark.git"
REPO_DIR  = Path("/content/edge-inference-benchmark")
DRIVE_BASE = Path("/content/drive/MyDrive/edge-inference-benchmark")

# Clone or pull repo so src/ modules are importable.
# Explicitly checks out develop — main only has the initial commit, all pipeline
# code (src/, scripts/, configs/) lives on the develop branch.
if not REPO_DIR.exists():
    !git clone -b develop "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" checkout develop
    !git -C "$REPO_DIR" pull --ff-only

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Copy ONNX model from Drive if not already on local disk
LOCAL_MODEL = REPO_DIR / "models/yolov8n.onnx"
LOCAL_MODEL.parent.mkdir(parents=True, exist_ok=True)

if not LOCAL_MODEL.exists():
    drive_model = DRIVE_BASE / "yolov8n.onnx"
    assert drive_model.exists(), (
        f"ONNX model not found at {drive_model}.\n"
        "Export locally: python scripts/export_model.py\n"
        "Then upload to Drive: MyDrive/edge-inference-benchmark/yolov8n.onnx"
    )
    shutil.copy(drive_model, LOCAL_MODEL)
    print(f"Model copied from Drive → {LOCAL_MODEL}")
else:
    print(f"Model already present: {LOCAL_MODEL}")

print(f"ONNX model size: {LOCAL_MODEL.stat().st_size / 1e6:.1f} MB")

# Sanity-check ONNX model via ORT — output must be (1, 84, 8400) for YOLOv8n at 640x640
_sess = ort.InferenceSession(str(LOCAL_MODEL), providers=["CUDAExecutionProvider"])
_input_name = _sess.get_inputs()[0].name
_dummy = np.zeros((1, 3, 640, 640), dtype=np.float32)
_out = _sess.run(None, {_input_name: _dummy})
assert _out[0].shape == (1, 84, 8400), (
    f"Unexpected ONNX output shape {_out[0].shape}; expected (1, 84, 8400). "
    "Re-export via scripts/export_model.py with opset=17, dynamic=False."
)
del _sess, _out
print(f"ONNX sanity check passed — output shape (1, 84, 8400). ✓")

ENGINE_DIR = REPO_DIR / "models"
ENGINE_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Cell 3 ── TensorRT FP32 Engine Build ─────────────────────────────────────
# Build and serialize a TensorRT FP32 engine from the ONNX model.
# Build time: ~1-2 minutes. Idempotent: skips if engine file already exists.

import tensorrt as trt


def build_trt_engine(
    onnx_path: str,
    engine_path: str,
    precision: str = "fp32",
    calibrator=None,
) -> str:
    """Build and serialize a TensorRT engine from an ONNX model.

    Args:
        onnx_path:    Path to source .onnx file (opset 17, static shape, no NMS).
        engine_path:  Destination path for the .engine file.
        precision:    'fp32', 'fp16', or 'int8'.
        calibrator:   IInt8EntropyCalibrator2 instance (int8 only).

    Returns:
        Absolute path to the written engine file.
    """
    trt_logger = trt.Logger(trt.Logger.WARNING)
    builder  = trt.Builder(trt_logger)
    network  = builder.create_network(
        trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
    )
    parser = trt.OnnxParser(network, trt_logger)

    with open(onnx_path, "rb") as f:
        raw = f.read()
    if not parser.parse(raw):
        errors = [str(parser.get_error(i)) for i in range(parser.num_errors)]
        raise RuntimeError("ONNX parse failed:\n" + "\n".join(errors))

    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)  # 1 GB

    if precision == "fp16":
        if not builder.platform_has_fast_fp16:
            raise RuntimeError("FP16 not supported on this GPU")
        config.set_flag(trt.BuilderFlag.FP16)
    elif precision == "int8":
        if not builder.platform_has_fast_int8:
            raise RuntimeError("INT8 not supported on this GPU")
        if calibrator is None:
            raise ValueError("calibrator is required for int8 precision")
        config.set_flag(trt.BuilderFlag.INT8)
        config.int8_calibrator = calibrator

    print(f"Building TensorRT {precision.upper()} engine — this may take 1-3 minutes...")
    engine_bytes = builder.build_serialized_network(network, config)
    if engine_bytes is None:
        raise RuntimeError(
            f"TensorRT engine build returned None (precision={precision}). "
            "Check nvidia-smi output and available VRAM."
        )

    with open(engine_path, "wb") as f:
        f.write(engine_bytes)

    size_mb = os.path.getsize(engine_path) / 1e6
    print(f"Engine written: {engine_path}  ({size_mb:.1f} MB)")
    return str(engine_path)


FP32_ENGINE = ENGINE_DIR / "yolov8n_fp32.engine"

if not FP32_ENGINE.exists():
    build_trt_engine(str(LOCAL_MODEL), str(FP32_ENGINE), precision="fp32")
else:
    print(f"FP32 engine already exists ({FP32_ENGINE.stat().st_size / 1e6:.1f} MB) — skipping build.")

In [ ]:
# ── Cell 4 ── TensorRT FP16 Engine Build ─────────────────────────────────────
# T4 has Turing Tensor Cores with native FP16 support — typical 2× speedup over FP32.
# Idempotent: skips if engine already exists.

FP16_ENGINE = ENGINE_DIR / "yolov8n_fp16.engine"

if not FP16_ENGINE.exists():
    build_trt_engine(str(LOCAL_MODEL), str(FP16_ENGINE), precision="fp16")
else:
    print(f"FP16 engine already exists ({FP16_ENGINE.stat().st_size / 1e6:.1f} MB) — skipping build.")

In [ ]:
# ── Cell 5 ── TensorRT INT8 Engine Build ─────────────────────────────────────
# Calibration uses 500 COCO val2017 images with letterbox preprocessing identical
# to inference — required for activation distribution to match the deployment path.
# Calibration cache is written to disk so rebuilding the engine skips recalibration.
# Note: calibration images drawn from val2017 — INT8 mAP may be ~0.001-0.002 optimistic.
# Idempotent: skips engine build if file exists; loads cache if calibration already ran.

import cv2
import json
import pycuda.driver as cuda
import pycuda.autoinit
import tensorrt as trt

CALIB_DIR   = REPO_DIR / "data/calibration"
INT8_ENGINE = ENGINE_DIR / "yolov8n_int8.engine"

# Copy calibration set from Drive if not already present on local disk
if not CALIB_DIR.exists() or not (CALIB_DIR / "manifest.json").exists():
    drive_calib = DRIVE_BASE / "calibration"
    assert drive_calib.exists(), (
        f"Calibration set not found locally or on Drive ({drive_calib}).\n"
        "Generate it: run scripts/run_benchmark.py once on Fedora (creates data/calibration/).\n"
        "Then upload data/calibration/ to Drive: MyDrive/edge-inference-benchmark/calibration/"
    )
    shutil.copytree(str(drive_calib), str(CALIB_DIR))
    n_imgs = len(list(CALIB_DIR.glob("*.jpg")))
    print(f"Calibration set copied from Drive: {n_imgs} images")
else:
    n_imgs = len(list(CALIB_DIR.glob("*.jpg")))
    print(f"Calibration set already present: {n_imgs} images")

assert n_imgs == 500, (
    f"Expected 500 calibration images, found {n_imgs}. "
    "Re-generate via run_benchmark.py with seed=42."
)


class CocoInt8Calibrator(trt.IInt8EntropyCalibrator2):
    """INT8 entropy calibrator over 500 letterbox-preprocessed COCO images.

    Applies the same letterbox preprocessing used at inference time so
    calibration activations match the deployment data distribution exactly.
    Image order is fixed by manifest.json (seed=42) for reproducibility.
    """

    def __init__(self, calib_dir: str) -> None:
        super().__init__()
        calib_path = Path(calib_dir)
        manifest   = json.loads((calib_path / "manifest.json").read_text())
        self._image_paths = [str(calib_path / name) for name in manifest["images"]]
        self._index       = 0
        n_bytes           = 1 * 3 * 640 * 640 * np.float32().itemsize
        self._device_buf  = cuda.mem_alloc(n_bytes)
        self._cache_path  = ENGINE_DIR / "int8_calibration.cache"
        print(f"Calibrator: {len(self._image_paths)} images, cache → {self._cache_path}")

    def get_batch_size(self) -> int:
        return 1

    def get_batch(self, names):
        """Feed one preprocessed image to TRT calibration at a time."""
        if self._index >= len(self._image_paths):
            return None
        from src.data.coco_loader import letterbox_preprocess
        bgr = cv2.imread(self._image_paths[self._index])
        if bgr is None:
            # Corrupted file — feed zeros; calibration is robust to a few bad images
            tensor = np.zeros((1, 3, 640, 640), dtype=np.float32)
        else:
            tensor, _ = letterbox_preprocess(bgr)
        cuda.memcpy_htod(self._device_buf, np.ascontiguousarray(tensor))
        self._index += 1
        if self._index % 100 == 0:
            print(f"  Calibrated {self._index}/{len(self._image_paths)} images...")
        return [int(self._device_buf)]

    def read_calibration_cache(self):
        if self._cache_path.exists():
            print("Loading calibration cache from disk (skipping re-calibration).")
            return self._cache_path.read_bytes()
        return None

    def write_calibration_cache(self, cache: bytes) -> None:
        self._cache_path.write_bytes(cache)
        print(f"Calibration cache written: {self._cache_path}")


if not INT8_ENGINE.exists():
    calibrator = CocoInt8Calibrator(str(CALIB_DIR))
    build_trt_engine(
        str(LOCAL_MODEL), str(INT8_ENGINE),
        precision="int8", calibrator=calibrator,
    )
else:
    print(f"INT8 engine already exists ({INT8_ENGINE.stat().st_size / 1e6:.1f} MB) — skipping build.")

In [ ]:
# ── Cell 6 ── Latency + Memory Benchmark ─────────────────────────────────────
# Protocol: 10 warmup passes (discarded), 100 timed passes per engine.
# Timing: time.perf_counter() — sub-millisecond resolution.
# Memory: torch.cuda.max_memory_allocated() captured after warmup (steady-state VRAM).
# GPU buffers are pinned (page-locked) to minimise PCIe transfer overhead.
# Idempotent: each run is stateless; GPU boost clock state may vary between sessions.

import time
import statistics
import pycuda.driver as cuda
import pycuda.autoinit
import tensorrt as trt

N_WARMUP     = 10    # discarded — populates GPU caches and JIT-compiled kernels
N_RUNS       = 100   # timed benchmark runs per benchmark protocol
INPUT_SHAPE  = (1, 3, 640, 640)
OUTPUT_SHAPE = (1, 84, 8400)


class TRTSession:
    """Minimal TensorRT inference session for latency benchmarking.

    Allocates pinned host buffers and CUDA device buffers once at construction.
    Inference is synchronous: async copy + execute_async_v2 + stream sync.
    """

    def __init__(self, engine_path: str) -> None:
        trt_logger = trt.Logger(trt.Logger.WARNING)
        runtime    = trt.Runtime(trt_logger)
        with open(engine_path, "rb") as f:
            self._engine = runtime.deserialize_cuda_engine(f.read())
        if self._engine is None:
            raise RuntimeError(
                f"Failed to deserialize TRT engine: {engine_path}\n"
                "Engine files are not portable across TRT versions or GPU generations. "
                "Re-run Cells 3-5 to rebuild."
            )
        self._context = self._engine.create_execution_context()

        # Verify output binding shape — catches wrong ONNX export silently
        out_shape = tuple(self._engine.get_binding_shape(1))
        if out_shape != OUTPUT_SHAPE:
            raise RuntimeError(
                f"TRT output binding shape {out_shape} != expected {OUTPUT_SHAPE}. "
                "Ensure yolov8n.onnx was exported with opset=17, dynamic=False, "
                "no NMS post-processing baked in."
            )

        n_in  = int(np.prod(INPUT_SHAPE))
        n_out = int(np.prod(OUTPUT_SHAPE))
        self._h_input  = cuda.pagelocked_empty(n_in,  dtype=np.float32)
        self._h_output = cuda.pagelocked_empty(n_out, dtype=np.float32)
        self._d_input  = cuda.mem_alloc(self._h_input.nbytes)
        self._d_output = cuda.mem_alloc(self._h_output.nbytes)
        self._stream   = cuda.Stream()

    def infer(self, input_tensor: np.ndarray) -> np.ndarray:
        """Synchronous inference. Input: (1,3,640,640) float32. Returns (1,84,8400)."""
        np.copyto(self._h_input, input_tensor.ravel())
        cuda.memcpy_htod_async(self._d_input, self._h_input, self._stream)
        self._context.execute_async_v2(
            bindings=[int(self._d_input), int(self._d_output)],
            stream_handle=self._stream.handle,
        )
        cuda.memcpy_dtoh_async(self._h_output, self._d_output, self._stream)
        self._stream.synchronize()
        return self._h_output.reshape(OUTPUT_SHAPE).copy()


ENGINES = {
    "tensorrt_fp32": str(FP32_ENGINE),
    "tensorrt_fp16": str(FP16_ENGINE),
    "tensorrt_int8": str(INT8_ENGINE),
}

latency_results = {}  # name → dict with timing stats + peak_memory_mb
dummy = np.random.default_rng(42).random(INPUT_SHAPE).astype(np.float32)

print(f"{'Runtime':25s}  {'mean':>8}  {'p95':>8}  {'stddev':>8}  {'fps':>7}  {'VRAM MB':>8}")
print("-" * 80)

for name, engine_path in ENGINES.items():
    # Reset VRAM peak counter before loading this session so we capture only
    # this engine's allocation, not carry-over from previous sessions.
    torch.cuda.reset_peak_memory_stats()

    session = TRTSession(engine_path)

    # Warmup: populates GPU kernel caches and primes the execution pipeline.
    # Memory snapshot is taken AFTER warmup so it reflects the steady-state VRAM
    # footprint, not transient first-inference allocations.
    for _ in range(N_WARMUP):
        session.infer(dummy)

    # Steady-state VRAM after warmup — the relevant deployment memory metric.
    # TRT pre-allocates all activation buffers at engine creation; inference
    # reuses them, so peak_mb is stable after warmup (delta ≈ 0 per call).
    peak_mb = torch.cuda.max_memory_allocated() / 1e6

    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        session.infer(dummy)
        t1 = time.perf_counter()
        latencies.append((t1 - t0) * 1000)

    mean_ms   = statistics.mean(latencies)
    stddev_ms = statistics.stdev(latencies)
    p95_ms    = float(np.percentile(latencies, 95))
    min_ms    = min(latencies)
    max_ms    = max(latencies)

    latency_results[name] = {
        "mean_ms":   mean_ms,
        "stddev_ms": stddev_ms,
        "p95_ms":    p95_ms,
        "min_ms":    min_ms,
        "max_ms":    max_ms,
        "n_runs":    N_RUNS,
        "n_warmup":  N_WARMUP,
        "peak_memory_mb": peak_mb,
    }

    print(
        f"{name:25s}  {mean_ms:>7.2f}ms  {p95_ms:>7.2f}ms  "
        f"{stddev_ms:>7.2f}ms  {1000/mean_ms:>6.1f}  {peak_mb:>7.1f}"
    )

    # Free CUDA resources before next engine to avoid VRAM fragmentation.
    del session
    torch.cuda.empty_cache()

print(f"\nBenchmark complete: {N_RUNS} timed runs per engine ({N_WARMUP} warmup discarded).")

In [ ]:
# ── Cell 7 ── Accuracy Evaluation (mAP@0.5:0.95) ─────────────────────────────
# Evaluates each TRT precision variant on the full COCO val2017 set (5000 images).
# Thresholds match the Fedora CPU runs for cross-runtime comparability:
#   eval_conf_threshold=0.001  — exposes the full PR curve to COCOeval
#   eval_iou_threshold=0.7     — matches ultralytics reference validator NMS
# mAP delta is always computed relative to TRT FP32 baseline, never cross-runtime.
# Runtime per precision: ~5 minutes inference + ~30s COCOeval = ~17 min total.
# Idempotent: COCOeval is stateless per call.

from src.benchmark.accuracy_evaluator import AccuracyResult, compute_map_delta, evaluate_map
from src.data.coco_loader import CocoLoader

# These thresholds must match eval_conf_threshold and eval_iou_threshold in
# configs/benchmark_config.yaml so TRT and CPU results are directly comparable.
EVAL_CONF_THRESHOLD = 0.001
EVAL_IOU_THRESHOLD  = 0.7

COCO_DIR    = REPO_DIR / "data/val2017"
ANNOTATIONS = REPO_DIR / "data/annotations/instances_val2017.json"

# Copy COCO data from Drive if not present on local disk
if not COCO_DIR.exists() or not ANNOTATIONS.exists():
    drive_val = DRIVE_BASE / "val2017"
    drive_ann = DRIVE_BASE / "annotations"
    assert drive_val.exists() and drive_ann.exists(), (
        f"COCO val2017 not found at {COCO_DIR} or Drive.\n"
        "Upload:\n"
        "  data/val2017/       → MyDrive/edge-inference-benchmark/val2017/\n"
        "  data/annotations/   → MyDrive/edge-inference-benchmark/annotations/"
    )
    print("Copying COCO val2017 from Drive (~1 GB, takes a few minutes)...")
    shutil.copytree(str(drive_val), str(COCO_DIR))
    shutil.copytree(str(drive_ann), str(REPO_DIR / "data/annotations"))
    print(f"COCO val2017 ready: {len(list(COCO_DIR.glob('*.jpg')))} images")

loader = CocoLoader(images_dir=str(COCO_DIR), annotations_file=str(ANNOTATIONS))
print(f"COCO val2017: {len(loader)} images loaded")
assert len(loader) == 5000, f"Expected 5000 images, got {len(loader)}"


class TRTRuntimeAdapter:
    """Wraps TRTSession to satisfy the runtime.infer interface expected by evaluate_map."""

    def __init__(self, session: TRTSession, name: str) -> None:
        self._session = session
        self.name     = name

    def infer(self, input_tensor: np.ndarray) -> np.ndarray:
        return self._session.infer(input_tensor)


accuracy_results = {}  # name → AccuracyResult
fp32_baseline    = None

for name, engine_path in ENGINES.items():
    print(f"\nEvaluating mAP@0.5:0.95 for {name} across 5000 images...")
    session = TRTSession(engine_path)
    adapter = TRTRuntimeAdapter(session, name)

    result = evaluate_map(
        adapter, loader, str(ANNOTATIONS),
        conf_threshold=EVAL_CONF_THRESHOLD,
        iou_threshold=EVAL_IOU_THRESHOLD,
    )
    accuracy_results[name] = result

    if name == "tensorrt_fp32":
        fp32_baseline = result

    del session
    torch.cuda.empty_cache()

print("\n=== mAP Results ===")
print(f"  eval_conf_threshold: {EVAL_CONF_THRESHOLD}  eval_iou_threshold: {EVAL_IOU_THRESHOLD}")
print(f"{'Runtime':25s}  {'mAP@50:95':>10}  {'mAP@50':>8}  {'delta vs FP32':>14}")
print("-" * 65)
for name, result in accuracy_results.items():
    delta = (
        0.0 if name == "tensorrt_fp32"
        else compute_map_delta(fp32_baseline, result)
    )
    print(f"{name:25s}  {result.map_50_95:>10.4f}  {result.map_50:>8.4f}  {delta:>+14.4f}")

In [ ]:
# ── Cell 8 ── Assemble Results + Export ───────────────────────────────────────
# Builds BenchmarkResult records (same schema as Fedora CPU runs), writes
# JSON (one per precision) and summary CSV, then backs up to Google Drive.
# Idempotent: files are overwritten on each run with deterministic content.

from src.results.result_schema import BenchmarkResult
from src.results.result_writer import ResultWriter
from src.utils.device_info import get_device_info

RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Enrich device_info with GPU details not captured by the CPU-oriented get_device_info()
device_info = get_device_info()
device_info["gpu"]           = torch.cuda.get_device_name(0)
device_info["gpu_memory_gb"] = round(
    torch.cuda.get_device_properties(0).total_memory / 1e9, 2
)
device_info["colab_env"] = COLAB_ENV

writer      = ResultWriter(output_dir=str(RESULTS_DIR))
all_results = []

precision_map = {
    "tensorrt_fp32": "fp32",
    "tensorrt_fp16": "fp16",
    "tensorrt_int8": "int8",
}

fp32_acc = accuracy_results.get("tensorrt_fp32")

for name, lat in latency_results.items():
    precision = precision_map[name]
    acc = accuracy_results.get(
        name,
        AccuracyResult(map_50_95=0.0, map_50=0.0, precision=precision, runtime=name),
    )
    delta = (
        0.0         if precision == "fp32" else
        compute_map_delta(fp32_acc, acc) if fp32_acc else float("nan")
    )

    # peak_memory_mb:       steady-state VRAM after warmup (measured in Cell 6)
    # peak_memory_delta_mb: TRT pre-allocates all activation buffers at engine
    #                       creation; each inference reuses them — marginal cost ≈ 0.
    #                       Analogous to onnx_cpu_fp32 delta_mb=0.0 on Fedora.
    result = BenchmarkResult(
        runtime              = name,
        precision            = precision,
        hardware             = "colab_t4",
        mean_latency_ms      = lat["mean_ms"],
        stddev_latency_ms    = lat["stddev_ms"],
        p95_latency_ms       = lat["p95_ms"],
        min_latency_ms       = lat["min_ms"],
        max_latency_ms       = lat["max_ms"],
        fps                  = 1000.0 / lat["mean_ms"],
        map_50_95            = acc.map_50_95,
        map_50               = acc.map_50,
        map_delta_vs_fp32    = delta,
        peak_memory_mb       = lat["peak_memory_mb"],
        peak_memory_delta_mb = 0.0,
        n_runs               = lat["n_runs"],
        n_warmup             = lat["n_warmup"],
        onnxruntime_version  = device_info.get("onnxruntime_version", "N/A"),
        torch_version        = torch.__version__,
        hardware_info        = device_info,
    )
    writer.write_json(result)
    all_results.append(result)
    print(f"Written: results/{name}.json  "
          f"mAP={acc.map_50_95:.4f}  latency={lat['mean_ms']:.2f}ms  "
          f"VRAM={lat['peak_memory_mb']:.0f}MB")

writer.write_csv(all_results)
print(f"Written: results/summary.csv ({len(all_results)} rows)")

# Back up result JSONs and CSV to Drive for persistent storage
drive_results = DRIVE_BASE / "results"
drive_results.mkdir(parents=True, exist_ok=True)
for f in RESULTS_DIR.glob("tensorrt_*.json"):
    shutil.copy(f, drive_results / f.name)
csv_src = RESULTS_DIR / "summary.csv"
if csv_src.exists():
    shutil.copy(csv_src, drive_results / "summary_colab_t4.csv")
print(f"\nResults backed up to Drive: {drive_results}")

print("\n=== TensorRT Benchmark Complete ===")
print(f"{'Runtime':25s}  {'mean':>8}  {'p95':>8}  {'fps':>7}  {'mAP':>7}  {'delta':>7}  {'VRAM':>8}")
print("-" * 85)
for r in all_results:
    delta_str = f"{r.map_delta_vs_fp32:+.4f}" if r.map_delta_vs_fp32 is not None else "  N/A  "
    print(
        f"{r.runtime:25s}  {r.mean_latency_ms:>7.2f}ms  {r.p95_latency_ms:>7.2f}ms  "
        f"{r.fps:>6.1f}  {r.map_50_95:>7.4f}  {delta_str:>7}  {r.peak_memory_mb:>7.0f}MB"
    )